# Conversational Ai aka ChatBot

In [28]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import json
import gradio as gr

In [29]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins with {openai_api_key[:8]}")
else:
    print("OpenAI API key not found")

OpenAI API Key exists and begins with sk-proj-


In [30]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [31]:
def chat(message, history):
    history = [{"role":h["role"], "content" : h["content"]} for h in history]
    messages = [{"role":"system", "content":system_message}] + history + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(model=MODEL, messages = messages)
    return response.choices[0].message.content

## Making the interaction streamable


In [33]:
def chat(message, history):
    history = [{"role":h["role"], "content" : h["content"]} for h in history]
    messages = [{"role":"system", "content":system_message}] + history + [{"role":"user", "content":message}]
    stream = openai.chat.completions.create(model=MODEL, messages = messages, stream = True)
    response=""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [34]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Adding tools for the Ticket Price context.


In [42]:
ticket_prices = {
    "london": "Rs. 75,513", 
    "paris": "Rs. 84,964", 
    "tokyo": "Rs. 132,314", 
    "berlin": "Rs. 47,160"
}

def get_ticket_price(destination_city):
    print(f"Tool is called for {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unkown Ticket Price")
    return f"The price of the ticket for {destination_city} is {price}"

In [43]:
get_ticket_price("LONDON")

Tool is called for LONDON


'The price of the ticket for LONDON is Rs. 75,513'

In [45]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [46]:
tools = [{"type": "function", "function": price_function}]

In [ ]:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [50]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [51]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


e:\Projects\flight-AI-assistant\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
e:\Projects\flight-AI-assistant\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
e:\Projects\flight-AI-assistant\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool is called for London


e:\Projects\flight-AI-assistant\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
